## Setup: paleta, librerías y funciones

In [1]:
BLUE='#1B3A6B'; FORE='#2E6DB4'; FIT='#3D85C8'; REAL='#5BA3D9'
GRAY='#92C4E8'; IC='#B5D8F2'; BG='#F4F7FB'; CARD='#FFFFFF'

import pandas as pd, numpy as np, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')

MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
fmt_M = mticker.FuncFormatter(lambda x, _: f'${x:.2f}M')

def grafica_pronostico_estilo(ts_full, fitted_vals, fc_mean, fc_ci,
                               etiqueta_y, titulo, nombre_archivo):
    fig, ax = plt.subplots(figsize=(18, 5), facecolor=BG)
    ax.set_facecolor(BG)
    ax.plot(ts_full.index, ts_full.values/1e6,
            color=BLUE, lw=1.8, label='Histórico', zorder=3)
    ax.plot(fitted_vals.index, fitted_vals.values/1e6,
            color=FORE, lw=1.1, linestyle='dotted', label='Ajuste modelo', zorder=2)
    ax.fill_between(fc_ci.index,
                    fc_ci.iloc[:,0]/1e6, fc_ci.iloc[:,1]/1e6,
                    color=IC, alpha=0.45, label='IC 95%', zorder=1)
    ax.plot(fc_mean.index, fc_mean.values/1e6,
            color=FORE, lw=2.2, linestyle='dashed', label='Pronóstico', zorder=4)
    ax.axvline(x=fc_mean.index[0], color=BLUE, lw=1.0, linestyle='--', alpha=0.55, zorder=5)
    ax.yaxis.set_major_formatter(fmt_M)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='both', labelsize=9)
    ax.set_ylabel(etiqueta_y, fontsize=10, color='#333')
    ax.set_title(titulo, fontsize=13, fontweight='bold', color=BLUE, pad=10)
    ax.grid(True, axis='y', linestyle='--', alpha=0.35, color=GRAY)
    ax.grid(True, axis='x', linestyle='--', alpha=0.20, color=GRAY)
    ax.spines[['top','right','left','bottom']].set_visible(False)
    ax.legend(loc='upper left', fontsize=9.5, framealpha=0.75, frameon=True, edgecolor='#ccc')
    plt.tight_layout()
    plt.savefig(nombre_archivo, dpi=180, bbox_inches='tight', facecolor=BG)
    plt.close()
    print(f'Guardado: {nombre_archivo}')

def tabla_pronostico(serie_forecast, titulo, nombre_archivo):
    n=len(serie_forecast); ROW_H=0.52; HDR_H=0.68; TIT_H=0.60; PAD=0.30
    COL_W=[2.6,3.6]; FIG_W=sum(COL_W)+2*PAD; FIG_H=TIT_H+HDR_H+n*ROW_H+2*PAD
    fig=plt.figure(figsize=(FIG_W,FIG_H),facecolor='white')
    ax=fig.add_axes([0,0,1,1]); ax.set_xlim(0,FIG_W); ax.set_ylim(0,FIG_H); ax.axis('off')
    x0,x1,x2=PAD,PAD+COL_W[0],PAD+sum(COL_W); xm1,xm2=(x0+x1)/2,(x1+x2)/2; y_top=FIG_H-PAD
    y_tit=y_top-TIT_H
    ax.add_patch(plt.Rectangle((x0,y_tit),sum(COL_W),TIT_H,color=BLUE,zorder=2))
    ax.text((x0+x2)/2,(y_top+y_tit)/2,titulo,ha='center',va='center',fontsize=10.5,fontweight='bold',color='white',zorder=3)
    y_hdr=y_tit-HDR_H
    for xL,xR,lbl in [(x0,x1,'MES'),(x1,x2,'PRONÓSTICO')]:
        ax.add_patch(plt.Rectangle((xL,y_hdr),xR-xL,HDR_H,color=BLUE,zorder=2))
        ax.text((xL+xR)/2,(y_tit+y_hdr)/2,lbl,ha='center',va='center',fontsize=10,fontweight='bold',color='white',zorder=3)
    ax.plot([x1,x1],[y_hdr,y_tit],color='white',lw=1.5,zorder=4)
    y_cur=y_hdr
    for i,(fecha,valor) in enumerate(serie_forecast.items()):
        y_bot=y_cur-ROW_H; fill='#D8EBF7' if i%2==0 else '#FFFFFF'
        ax.add_patch(plt.Rectangle((x0,y_bot),sum(COL_W),ROW_H,color=fill,zorder=2))
        ax.plot([x1,x1],[y_bot,y_cur],color='#B0C8E0',lw=0.8,zorder=3)
        ax.plot([x0,x2],[y_bot,y_bot],color='#B0C8E0',lw=0.4,zorder=3)
        yc=(y_cur+y_bot)/2
        ax.text(xm1,yc,fecha.strftime('%Y-%m'),ha='center',va='center',fontsize=9.5,fontweight='bold',color=BLUE,zorder=4)
        ax.text(xm2,yc,f'${valor:,.0f} MXN',ha='center',va='center',fontsize=9.5,fontweight='bold',color='#1A1A2E',zorder=4)
        y_cur=y_bot
    ax.add_patch(plt.Rectangle((x0,y_cur),sum(COL_W),FIG_H-PAD-y_cur,fill=False,edgecolor=BLUE,lw=2.0,zorder=5))
    plt.savefig(nombre_archivo,dpi=180,bbox_inches='tight',facecolor='white')
    plt.close()
    print(f'Guardado: {nombre_archivo}')

print('Setup completo.')

Setup completo.


## Carga del dataset

In [2]:
df = pd.read_csv('Gastos.csv', encoding='latin1')
df.rename(columns={df.columns[0]: 'Fecha'}, inplace=True)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df_idx = df.set_index('Fecha')
print('Columnas:', df.columns.tolist())

Columnas: ['Fecha', 'Costo de Venta (Refacciones)', 'Costo de llantas', 'Costo de Lubricantes', 'Gastos de OperaciÃ³n', 'Gastos de Mantenimiento', 'Gastos Administrativos', 'Gastos Financieros', 'Gastos no fiscales', 'ISR  Ejercicio', 'ISR Facilidades Administrativas', 'Costo de lo Vendido']


## PARTE 1 — Gastos Operativos

In [3]:
col_refac  = [c for c in df_idx.columns if 'Costo de Venta (Refacciones)' in c][0]
col_llanta = [c for c in df_idx.columns if 'Costo de llantas' in c][0]
col_lubric = [c for c in df_idx.columns if 'Costo de Lubricantes' in c][0]
col_op     = [c for c in df_idx.columns if 'Gastos de Operaci' in c][0]

df_idx['Gastos Operativos'] = df_idx[col_refac] + df_idx[col_llanta] + df_idx[col_lubric] + df_idx[col_op]
ts_op = df_idx['Gastos Operativos'].asfreq('MS')

if ts_op['2023-01-01'] < 0:
    ts_op['2023-01-01'] = (ts_op['2022-12-01'] + ts_op['2023-02-01']) / 2

# Serie recortada a partir de 2023
ts_op_trim = ts_op['2023':]

train_op = ts_op_trim['2023-01':'2025-09']   # ene 2023 – sep 2025 (33 meses)
test_op  = ts_op_trim['2025-10':'2025-12']   # oct – dic 2025  (3 meses)

print(f'Train: {train_op.index[0].date()} → {train_op.index[-1].date()} ({len(train_op)} meses)')
print(f'Test : {test_op.index[0].date()} → {test_op.index[-1].date()} ({len(test_op)} meses)')

Train: 2023-01-01 → 2025-09-01 (33 meses)
Test : 2025-10-01 → 2025-12-01 (3 meses)


In [4]:
print('Ajustando SARIMA(1,1,1)(1,0,1)[12] — Gastos Operativos...')
model_op  = SARIMAX(train_op, order=(1,1,1), seasonal_order=(1,0,1,12),
                    enforce_stationarity=False, enforce_invertibility=False)
result_op = model_op.fit(disp=False, maxiter=300)
print(result_op.summary())

# Validación: 3 pasos (oct–dic 2025)
fc_op_val  = result_op.get_forecast(steps=3)
fc_op_mean = fc_op_val.predicted_mean
fc_op_ci   = fc_op_val.conf_int(alpha=0.05)
fc_op_mean.index = test_op.index
fc_op_ci.index   = test_op.index

mae_op  = mean_absolute_error(test_op, fc_op_mean)
rmse_op = np.sqrt(mean_squared_error(test_op, fc_op_mean))
mape_op = np.mean(np.abs((test_op - fc_op_mean) / test_op)) * 100
print(f'MAE: ${mae_op:,.0f} | RMSE: ${rmse_op:,.0f} | MAPE: {mape_op:.1f}%')

fitted_op = result_op.fittedvalues
decomp_op = seasonal_decompose(train_op, model='additive', period=12)
sea_op     = pd.Series(decomp_op.seasonal.values, index=decomp_op.seasonal.index)
sea_op_mes = sea_op.groupby(sea_op.index.month).mean()
sea_op_mes.index = MESES

# Reentrenar con serie completa recortada (2023–2025)
model_op2  = SARIMAX(ts_op_trim, order=(1,1,1), seasonal_order=(1,0,1,12),
                     enforce_stationarity=False, enforce_invertibility=False)
res_op2    = model_op2.fit(disp=False, maxiter=300)
fc_op2_obj = res_op2.get_forecast(steps=12)
fc_op2     = fc_op2_obj.predicted_mean
fc_op2_ci  = fc_op2_obj.conf_int(alpha=0.05)
fc_op2.index    = pd.date_range('2026-01-01', periods=12, freq='MS')
fc_op2_ci.index = fc_op2.index

fitted_op_full = res_op2.fittedvalues
print('Modelos Gastos Operativos listos.')

Ajustando SARIMA(1,1,1)(1,0,1)[12] — Gastos Operativos...
                                     SARIMAX Results                                      
Dep. Variable:                  Gastos Operativos   No. Observations:                   33
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 12)   Log Likelihood                -286.227
Date:                            Wed, 06 May 2026   AIC                            582.453
Time:                                    19:56:32   BIC                            586.905
Sample:                                01-01-2023   HQIC                           583.067
                                     - 09-01-2025                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1274      1.920      0.066      0.947  

### Gráfica principal — estilo referencia (histórico + ajuste + IC + proyección)

In [5]:
grafica_pronostico_estilo(
    ts_full        = ts_op_trim,
    fitted_vals    = fitted_op_full,
    fc_mean        = fc_op2,
    fc_ci          = fc_op2_ci,
    etiqueta_y     = 'Millones MXN',
    titulo         = 'SARIMA(1,1,1)(1,0,1)[12] — Gastos Operativos · Pronóstico 2026',
    nombre_archivo = 'pronostico_gastos_operativos.png'
)

Guardado: pronostico_gastos_operativos.png


### Dashboard 5 paneles + Tabla estilizada

In [6]:
fig=plt.figure(figsize=(20,22),facecolor=BG)
fig.suptitle('SARIMA(1,1,1)(1,0,1)[12] — Gastos Operativos\n(Refacciones + Llantas + Lubricantes + Operación)',
             fontsize=19,fontweight='bold',color=BLUE,y=0.987)
gs=fig.add_gridspec(4,2,hspace=0.52,wspace=0.30,left=0.08,right=0.96,top=0.96,bottom=0.04)

# ── Panel 1: histórico + ajuste (train 2023–sep 2025) ──
ax1=fig.add_subplot(gs[0,:]); ax1.set_facecolor(CARD)
ax1.plot(train_op.index,train_op.values/1e6,color=GRAY,lw=1.5,alpha=0.9,label='Real (train 2023–sep 2025)')
ax1.plot(fitted_op.index,fitted_op.values/1e6,color=BLUE,lw=1.3,alpha=0.75,label='Ajuste in-sample')
ax1.set_title('Serie histórica Gastos Operativos con ajuste del modelo',fontsize=12,fontweight='bold',color=BLUE,pad=8)
ax1.set_ylabel('Millones MXN',fontsize=10); ax1.yaxis.set_major_formatter(fmt_M)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y')); ax1.tick_params(axis='x',labelsize=8)
ax1.legend(fontsize=9,loc='upper left',framealpha=0.7)
ax1.grid(axis='y',linestyle='--',alpha=0.35); ax1.spines[['top','right']].set_visible(False)
for yr in ['2024-01-01','2025-01-01']:
    ax1.axvline(pd.Timestamp(yr),color=BLUE,lw=0.6,linestyle=':',alpha=0.3)

# ── Panel 2: validación oct–dic 2025 ──
ax2=fig.add_subplot(gs[1,:]); ax2.set_facecolor(CARD)
ctx=train_op[-6:]
ax2.plot(ctx.index,ctx.values/1e6,color=GRAY,lw=1.3,alpha=0.5,label='Real (abr–sep 2025)')
ax2.fill_between(fc_op_ci.index,fc_op_ci.iloc[:,0]/1e6,fc_op_ci.iloc[:,1]/1e6,color=FORE,alpha=0.18,label='IC 95%')
ax2.plot(fc_op_mean.index,fc_op_mean.values/1e6,color=FORE,lw=2.3,linestyle='--',label='Pronóstico SARIMA')
ax2.plot(test_op.index,test_op.values/1e6,color=BLUE,lw=2.0,marker='o',markersize=6,label='Real oct–dic 2025')
ax2.set_title('Pronóstico SARIMA vs Real 2025 — Gastos Operativos (validación oct–dic)',fontsize=12,fontweight='bold',color=BLUE,pad=8)
ax2.set_ylabel('Millones MXN',fontsize=10); ax2.yaxis.set_major_formatter(fmt_M)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y')); ax2.tick_params(axis='x',labelsize=8)
ax2.legend(fontsize=9,framealpha=0.8,loc='upper right')
ax2.grid(axis='y',linestyle='--',alpha=0.35); ax2.spines[['top','right']].set_visible(False)
ax2.text(0.01,0.97,f'MAE=${mae_op/1e6:.2f}M  RMSE=${rmse_op/1e6:.2f}M  MAPE={mape_op:.1f}%',
         transform=ax2.transAxes,ha='left',fontsize=9.5,va='top',
         bbox=dict(boxstyle='round,pad=0.35',fc=IC,ec=FORE,lw=1.2))

# ── Panel 3: estacionalidad ──
ax3=fig.add_subplot(gs[2,0]); ax3.set_facecolor(CARD)
bc_op=[FORE if v==sea_op_mes.min() else REAL if v==sea_op_mes.max() else BLUE for v in sea_op_mes.values]
bars_op=ax3.bar(sea_op_mes.index,sea_op_mes.values/1e6,color=bc_op,edgecolor='white',linewidth=0.6,width=0.65,zorder=3)
ax3.axhline(0,color=GRAY,lw=1,linestyle='--')
ax3.set_title('Componente estacional promedio\npor mes del año',fontsize=11,fontweight='bold',color=BLUE,pad=8)
ax3.set_ylabel('Desviación vs tendencia (M MXN)',fontsize=10); ax3.yaxis.set_major_formatter(fmt_M)
ax3.grid(axis='y',linestyle='--',alpha=0.4,zorder=0); ax3.spines[['top','right']].set_visible(False)
for bar,val in zip(bars_op,sea_op_mes.values):
    ax3.text(bar.get_x()+bar.get_width()/2,val/1e6+(0.08 if val>=0 else -0.22),
             f'${val/1e6:.1f}M',ha='center',va='bottom',fontsize=7.5,fontweight='bold',color='#222')
ax3.text(0.5,-0.18,f'Mes con mayor gasto estacional: {sea_op_mes.idxmax()}',
         transform=ax3.transAxes,ha='center',fontsize=10,fontweight='bold',color=FORE)

# ── Panel 4: residuos (3 meses) ──
resid_op=test_op-fc_op_mean
ax4=fig.add_subplot(gs[2,1]); ax4.set_facecolor(CARD)
rc_op=[REAL if r>=0 else FORE for r in resid_op.values]
ax4.bar(range(len(resid_op)),resid_op.values/1e6,color=rc_op,edgecolor='white',linewidth=0.4,width=0.65,zorder=3)
ax4.axhline(0,color=GRAY,lw=1,linestyle='--')
ax4.set_xticks(range(len(resid_op)))
ax4.set_xticklabels([d.strftime('%b\n%Y') for d in resid_op.index],fontsize=9)
ax4.set_title('Error del pronóstico\n(Real − Pronóstico) oct–dic 2025',fontsize=11,fontweight='bold',color=BLUE,pad=8)
ax4.set_ylabel('Error (M MXN)',fontsize=10); ax4.yaxis.set_major_formatter(fmt_M)
ax4.grid(axis='y',linestyle='--',alpha=0.35,zorder=0); ax4.spines[['top','right']].set_visible(False)
ax4.legend(handles=[Patch(color=REAL,label='Real > Pronóstico'),Patch(color=FORE,label='Real < Pronóstico')],fontsize=8,framealpha=0.7)

# ── Panel 5: pronóstico 2026 ──
ax5=fig.add_subplot(gs[3,:]); ax5.set_facecolor(CARD)
ctx2=ts_op_trim[-12:]
ax5.plot(ctx2.index,ctx2.values/1e6,color=GRAY,lw=1.5,alpha=0.7,label='Real 2025')
ax5.fill_between(fc_op2_ci.index,fc_op2_ci.iloc[:,0]/1e6,fc_op2_ci.iloc[:,1]/1e6,color=IC,alpha=0.15,label='IC 95%')
ax5.plot(fc_op2.index,fc_op2.values/1e6,color=FORE,lw=2.3,linestyle='--',marker='o',markersize=5,label='Pronóstico 2026')
ax5.axvline(pd.Timestamp('2026-01-01'),color=GRAY,lw=1,linestyle=':',alpha=0.6)
ax5.set_title('Pronóstico Gastos Operativos 2026 (modelo 2023–2025)',fontsize=12,fontweight='bold',color=BLUE,pad=8)
ax5.set_ylabel('Millones MXN',fontsize=10); ax5.yaxis.set_major_formatter(fmt_M)
ax5.xaxis.set_major_locator(mdates.MonthLocator())
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y')); ax5.tick_params(axis='x',labelsize=8)
ax5.legend(fontsize=9,framealpha=0.8)
ax5.grid(axis='y',linestyle='--',alpha=0.35); ax5.spines[['top','right']].set_visible(False)
for date,val in fc_op2.items():
    ax5.text(date,val/1e6+0.2,f'${val/1e6:.1f}M',ha='center',va='bottom',fontsize=7.5,color=REAL,fontweight='bold')

plt.savefig('gastos_operativos_dashboard.png',dpi=150,bbox_inches='tight',facecolor=BG)
plt.close(); print('Guardado: gastos_operativos_dashboard.png')

Guardado: gastos_operativos_dashboard.png


In [7]:
tabla_pronostico(fc_op2,'PRONÓSTICO GASTOS OPERATIVOS 2026','tabla_gastos_operativos_2026.png')
print('\n── Pronóstico Gastos Operativos 2026 ──')
for d,v in fc_op2.items(): print(f'  {d.strftime("%Y-%m")}  → ${v:>14,.0f} MXN')
print(f'\n  Total 2026: ${fc_op2.sum():>14,.0f} MXN | AIC: {res_op2.aic:.2f} | BIC: {res_op2.bic:.2f}')

Guardado: tabla_gastos_operativos_2026.png

── Pronóstico Gastos Operativos 2026 ──
  2026-01  → $    17,263,479 MXN
  2026-02  → $    16,976,390 MXN
  2026-03  → $    17,361,308 MXN
  2026-04  → $    16,950,946 MXN
  2026-05  → $    16,940,045 MXN
  2026-06  → $    17,202,179 MXN
  2026-07  → $    17,282,998 MXN
  2026-08  → $    17,261,933 MXN
  2026-09  → $    17,212,119 MXN
  2026-10  → $    17,423,797 MXN
  2026-11  → $    17,480,784 MXN
  2026-12  → $    17,129,803 MXN

  Total 2026: $   206,485,780 MXN | AIC: 676.56 | BIC: 681.78


## PARTE 2 — Gastos Administrativos

In [8]:
col_mant   = [c for c in df_idx.columns if 'Gastos de Mantenimiento' in c][0]
col_admin  = [c for c in df_idx.columns if 'Gastos Administrativos' in c][0]
col_fin    = [c for c in df_idx.columns if 'Gastos Financieros' in c][0]
col_nf     = [c for c in df_idx.columns if 'Gastos no fiscales' in c][0]
col_isr    = [c for c in df_idx.columns if 'ISR  Ejercicio' in c][0]
col_isr_fa = [c for c in df_idx.columns if 'ISR Facilidades Administrativas' in c][0]

df_idx['Gastos Administrativos'] = (df_idx[col_mant] + df_idx[col_admin] + df_idx[col_fin]
                                     + df_idx[col_nf] + df_idx[col_isr] + df_idx[col_isr_fa])
ts_adm = df_idx['Gastos Administrativos'].asfreq('MS')

if ts_adm['2023-01-01'] < 0:
    ts_adm['2023-01-01'] = (ts_adm['2022-12-01'] + ts_adm['2023-02-01']) / 2

# Serie recortada a partir de 2023
ts_adm_trim = ts_adm['2023':]

train_adm = ts_adm_trim['2023-01':'2025-09']   # ene 2023 – sep 2025 (33 meses)
test_adm  = ts_adm_trim['2025-10':'2025-12']   # oct – dic 2025  (3 meses)

print(f'Train: {train_adm.index[0].date()} → {train_adm.index[-1].date()} ({len(train_adm)} meses)')
print(f'Test : {test_adm.index[0].date()} → {test_adm.index[-1].date()} ({len(test_adm)} meses)')

Train: 2023-01-01 → 2025-09-01 (33 meses)
Test : 2025-10-01 → 2025-12-01 (3 meses)


In [9]:
print('Ajustando SARIMA(1,1,1)(1,0,1)[12] — Gastos Administrativos...')
model_adm  = SARIMAX(train_adm, order=(1,1,1), seasonal_order=(1,0,1,12),
                     enforce_stationarity=False, enforce_invertibility=False)
result_adm = model_adm.fit(disp=False, maxiter=300)
print(result_adm.summary())

# Validación: 3 pasos (oct–dic 2025)
fc_adm_val  = result_adm.get_forecast(steps=3)
fc_adm_mean = fc_adm_val.predicted_mean
fc_adm_ci   = fc_adm_val.conf_int(alpha=0.05)
fc_adm_mean.index = test_adm.index
fc_adm_ci.index   = test_adm.index

mae_adm  = mean_absolute_error(test_adm, fc_adm_mean)
rmse_adm = np.sqrt(mean_squared_error(test_adm, fc_adm_mean))
mape_adm = np.mean(np.abs((test_adm - fc_adm_mean) / test_adm)) * 100
print(f'MAE: ${mae_adm:,.0f} | RMSE: ${rmse_adm:,.0f} | MAPE: {mape_adm:.1f}%')

fitted_adm = result_adm.fittedvalues
decomp_adm = seasonal_decompose(train_adm, model='additive', period=12)
sea_adm     = pd.Series(decomp_adm.seasonal.values, index=decomp_adm.seasonal.index)
sea_adm_mes = sea_adm.groupby(sea_adm.index.month).mean()
sea_adm_mes.index = MESES

# Reentrenar con serie completa recortada (2023–2025)
model_adm2  = SARIMAX(ts_adm_trim, order=(1,1,1), seasonal_order=(1,0,1,12),
                      enforce_stationarity=False, enforce_invertibility=False)
res_adm2    = model_adm2.fit(disp=False, maxiter=300)
fc_adm2_obj = res_adm2.get_forecast(steps=12)
fc_adm2     = fc_adm2_obj.predicted_mean
fc_adm2_ci  = fc_adm2_obj.conf_int(alpha=0.05)
fc_adm2.index    = pd.date_range('2026-01-01', periods=12, freq='MS')
fc_adm2_ci.index = fc_adm2.index

fitted_adm_full = res_adm2.fittedvalues
print('Modelos Gastos Administrativos listos.')

Ajustando SARIMA(1,1,1)(1,0,1)[12] — Gastos Administrativos...
                                     SARIMAX Results                                      
Dep. Variable:             Gastos Administrativos   No. Observations:                   33
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 12)   Log Likelihood                -295.774
Date:                            Wed, 06 May 2026   AIC                            601.547
Time:                                    19:56:42   BIC                            605.999
Sample:                                01-01-2023   HQIC                           602.161
                                     - 09-01-2025                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2926      0.140      2.092      0.

### Gráfica principal — estilo referencia (histórico + ajuste + IC + proyección)

In [10]:
grafica_pronostico_estilo(
    ts_full        = ts_adm_trim,
    fitted_vals    = fitted_adm_full,
    fc_mean        = fc_adm2,
    fc_ci          = fc_adm2_ci,
    etiqueta_y     = 'Millones MXN',
    titulo         = 'SARIMA(1,1,1)(1,0,1)[12] — Gastos Administrativos · Pronóstico 2026',
    nombre_archivo = 'pronostico_gastos_administrativos.png'
)

Guardado: pronostico_gastos_administrativos.png


### Dashboard 5 paneles + Tabla estilizada

In [11]:
fig=plt.figure(figsize=(20,22),facecolor=BG)
fig.suptitle('SARIMA(1,1,1)(1,0,1)[12] — Gastos Administrativos\n(Mantenimiento + Administrativos + Financieros + No fiscales + ISR)',
             fontsize=19,fontweight='bold',color=BLUE,y=0.987)
gs=fig.add_gridspec(4,2,hspace=0.52,wspace=0.30,left=0.08,right=0.96,top=0.96,bottom=0.04)

# ── Panel 1: histórico + ajuste (train 2023–sep 2025) ──
ax1=fig.add_subplot(gs[0,:]); ax1.set_facecolor(CARD)
ax1.plot(train_adm.index,train_adm.values/1e6,color=GRAY,lw=1.5,alpha=0.9,label='Real (train 2023–sep 2025)')
ax1.plot(fitted_adm.index,fitted_adm.values/1e6,color=BLUE,lw=1.3,alpha=0.75,label='Ajuste in-sample')
ax1.set_title('Serie histórica Gastos Administrativos con ajuste del modelo',fontsize=12,fontweight='bold',color=BLUE,pad=8)
ax1.set_ylabel('Millones MXN',fontsize=10); ax1.yaxis.set_major_formatter(fmt_M)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y')); ax1.tick_params(axis='x',labelsize=8)
ax1.legend(fontsize=9,loc='upper left',framealpha=0.7)
ax1.grid(axis='y',linestyle='--',alpha=0.35); ax1.spines[['top','right']].set_visible(False)
for yr in ['2024-01-01','2025-01-01']:
    ax1.axvline(pd.Timestamp(yr),color=BLUE,lw=0.6,linestyle=':',alpha=0.3)

# ── Panel 2: validación oct–dic 2025 ──
ax2=fig.add_subplot(gs[1,:]); ax2.set_facecolor(CARD)
ctx=train_adm[-6:]
ax2.plot(ctx.index,ctx.values/1e6,color=GRAY,lw=1.3,alpha=0.5,label='Real (abr–sep 2025)')
ax2.fill_between(fc_adm_ci.index,fc_adm_ci.iloc[:,0]/1e6,fc_adm_ci.iloc[:,1]/1e6,color=FORE,alpha=0.18,label='IC 95%')
ax2.plot(fc_adm_mean.index,fc_adm_mean.values/1e6,color=FORE,lw=2.3,linestyle='--',label='Pronóstico SARIMA')
ax2.plot(test_adm.index,test_adm.values/1e6,color=BLUE,lw=2.0,marker='o',markersize=6,label='Real oct–dic 2025')
ax2.set_title('Pronóstico SARIMA vs Real 2025 — Gastos Administrativos (validación oct–dic)',fontsize=12,fontweight='bold',color=BLUE,pad=8)
ax2.set_ylabel('Millones MXN',fontsize=10); ax2.yaxis.set_major_formatter(fmt_M)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y')); ax2.tick_params(axis='x',labelsize=8)
ax2.legend(fontsize=9,framealpha=0.8,loc='upper right')
ax2.grid(axis='y',linestyle='--',alpha=0.35); ax2.spines[['top','right']].set_visible(False)
ax2.text(0.01,0.97,f'MAE=${mae_adm/1e6:.2f}M  RMSE=${rmse_adm/1e6:.2f}M  MAPE={mape_adm:.1f}%',
         transform=ax2.transAxes,ha='left',fontsize=9.5,va='top',
         bbox=dict(boxstyle='round,pad=0.35',fc=IC,ec=FORE,lw=1.2))

# ── Panel 3: estacionalidad ──
ax3=fig.add_subplot(gs[2,0]); ax3.set_facecolor(CARD)
bc_adm=[FORE if v==sea_adm_mes.min() else REAL if v==sea_adm_mes.max() else BLUE for v in sea_adm_mes.values]
bars_adm=ax3.bar(sea_adm_mes.index,sea_adm_mes.values/1e6,color=bc_adm,edgecolor='white',linewidth=0.6,width=0.65,zorder=3)
ax3.axhline(0,color=GRAY,lw=1,linestyle='--')
ax3.set_title('Componente estacional promedio\npor mes del año',fontsize=11,fontweight='bold',color=BLUE,pad=8)
ax3.set_ylabel('Desviación vs tendencia (M MXN)',fontsize=10); ax3.yaxis.set_major_formatter(fmt_M)
ax3.grid(axis='y',linestyle='--',alpha=0.4,zorder=0); ax3.spines[['top','right']].set_visible(False)
for bar,val in zip(bars_adm,sea_adm_mes.values):
    ax3.text(bar.get_x()+bar.get_width()/2,val/1e6+(0.08 if val>=0 else -0.22),
             f'${val/1e6:.1f}M',ha='center',va='bottom',fontsize=7.5,fontweight='bold',color='#222')
ax3.text(0.5,-0.18,f'Mes con mayor gasto estacional: {sea_adm_mes.idxmax()}',
         transform=ax3.transAxes,ha='center',fontsize=10,fontweight='bold',color=FORE)

# ── Panel 4: residuos (3 meses) ──
resid_adm=test_adm-fc_adm_mean
ax4=fig.add_subplot(gs[2,1]); ax4.set_facecolor(CARD)
rc_adm=[REAL if r>=0 else FORE for r in resid_adm.values]
ax4.bar(range(len(resid_adm)),resid_adm.values/1e6,color=rc_adm,edgecolor='white',linewidth=0.4,width=0.65,zorder=3)
ax4.axhline(0,color=GRAY,lw=1,linestyle='--')
ax4.set_xticks(range(len(resid_adm)))
ax4.set_xticklabels([d.strftime('%b\n%Y') for d in resid_adm.index],fontsize=9)
ax4.set_title('Error del pronóstico\n(Real − Pronóstico) oct–dic 2025',fontsize=11,fontweight='bold',color=BLUE,pad=8)
ax4.set_ylabel('Error (M MXN)',fontsize=10); ax4.yaxis.set_major_formatter(fmt_M)
ax4.grid(axis='y',linestyle='--',alpha=0.35,zorder=0); ax4.spines[['top','right']].set_visible(False)
ax4.legend(handles=[Patch(color=REAL,label='Real > Pronóstico'),Patch(color=FORE,label='Real < Pronóstico')],fontsize=8,framealpha=0.7)

# ── Panel 5: pronóstico 2026 ──
ax5=fig.add_subplot(gs[3,:]); ax5.set_facecolor(CARD)
ctx2=ts_adm_trim[-12:]
ax5.plot(ctx2.index,ctx2.values/1e6,color=GRAY,lw=1.5,alpha=0.7,label='Real 2025')
ax5.fill_between(fc_adm2_ci.index,fc_adm2_ci.iloc[:,0]/1e6,fc_adm2_ci.iloc[:,1]/1e6,color=IC,alpha=0.15,label='IC 95%')
ax5.plot(fc_adm2.index,fc_adm2.values/1e6,color=FORE,lw=2.3,linestyle='--',marker='o',markersize=5,label='Pronóstico 2026')
ax5.axvline(pd.Timestamp('2026-01-01'),color=GRAY,lw=1,linestyle=':',alpha=0.6)
ax5.set_title('Pronóstico Gastos Administrativos 2026 (modelo 2023–2025)',fontsize=12,fontweight='bold',color=BLUE,pad=8)
ax5.set_ylabel('Millones MXN',fontsize=10); ax5.yaxis.set_major_formatter(fmt_M)
ax5.xaxis.set_major_locator(mdates.MonthLocator())
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y')); ax5.tick_params(axis='x',labelsize=8)
ax5.legend(fontsize=9,framealpha=0.8)
ax5.grid(axis='y',linestyle='--',alpha=0.35); ax5.spines[['top','right']].set_visible(False)
for date,val in fc_adm2.items():
    ax5.text(date,val/1e6+0.2,f'${val/1e6:.1f}M',ha='center',va='bottom',fontsize=7.5,color=REAL,fontweight='bold')

plt.savefig('gastos_administrativos_dashboard.png',dpi=150,bbox_inches='tight',facecolor=BG)
plt.close(); print('Guardado: gastos_administrativos_dashboard.png')

Guardado: gastos_administrativos_dashboard.png


In [12]:
tabla_pronostico(fc_adm2,'PRONÓSTICO GASTOS ADMINISTRATIVOS 2026','tabla_gastos_administrativos_2026.png')
print('\n── Pronóstico Gastos Administrativos 2026 ──')
for d,v in fc_adm2.items(): print(f'  {d.strftime("%Y-%m")}  → ${v:>14,.0f} MXN')
print(f'\n  Total 2026: ${fc_adm2.sum():>14,.0f} MXN | AIC: {res_adm2.aic:.2f} | BIC: {res_adm2.bic:.2f}')

Guardado: tabla_gastos_administrativos_2026.png

── Pronóstico Gastos Administrativos 2026 ──
  2026-01  → $    29,667,964 MXN
  2026-02  → $    28,228,966 MXN
  2026-03  → $    28,000,799 MXN
  2026-04  → $    27,999,053 MXN
  2026-05  → $    34,739,450 MXN
  2026-06  → $    30,224,762 MXN
  2026-07  → $    29,968,706 MXN
  2026-08  → $    30,654,909 MXN
  2026-09  → $    29,876,199 MXN
  2026-10  → $    32,807,050 MXN
  2026-11  → $    31,036,992 MXN
  2026-12  → $    36,671,925 MXN

  Total 2026: $   369,876,775 MXN | AIC: 706.48 | BIC: 711.70


## PARTE 3 — Consolidado

In [13]:
fig,ax=plt.subplots(figsize=(16,6),facecolor=BG); ax.set_facecolor(CARD)
x=np.arange(12); w=0.35
b1=ax.bar(x-w/2,fc_op2.values/1e6,width=w,color=BLUE,edgecolor='white',linewidth=0.5,label='Operativos',zorder=3)
b2=ax.bar(x+w/2,fc_adm2.values/1e6,width=w,color=FORE,edgecolor='white',linewidth=0.5,label='Administrativos',zorder=3)
ax.plot(x,(fc_op2.values+fc_adm2.values)/1e6,color=REAL,marker='D',markersize=6,lw=2,label='Total Gastos',zorder=4)
ax.set_xticks(x); ax.set_xticklabels(MESES,fontsize=10)
ax.yaxis.set_major_formatter(fmt_M)
ax.set_title('Pronóstico Gastos 2026 — Operativos vs Administrativos',fontsize=14,fontweight='bold',color=BLUE,pad=10)
ax.set_ylabel('Millones MXN',fontsize=11)
ax.legend(fontsize=10,framealpha=0.85)
ax.grid(axis='y',linestyle='--',alpha=0.35,zorder=0); ax.spines[['top','right']].set_visible(False)
for bar in list(b1)+list(b2):
    h=bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2,h+0.15,f'${h:.1f}M',ha='center',va='bottom',fontsize=6.5,color='#333')
plt.tight_layout()
plt.savefig('gastos_consolidado_barras_2026.png',dpi=150,bbox_inches='tight',facecolor=BG)
plt.close(); print('Guardado: gastos_consolidado_barras_2026.png')
print(f'\nTotal Operativos 2026:      ${fc_op2.sum():>14,.0f} MXN')
print(f'Total Administrativos 2026: ${fc_adm2.sum():>14,.0f} MXN')
print(f'TOTAL GASTOS 2026:          ${fc_op2.sum()+fc_adm2.sum():>14,.0f} MXN')

Guardado: gastos_consolidado_barras_2026.png

Total Operativos 2026:      $   206,485,780 MXN
Total Administrativos 2026: $   369,876,775 MXN
TOTAL GASTOS 2026:          $   576,362,555 MXN
